# Week 5: LLM APIs and Production Use

**Applied Generative AI**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

*Developed by Prachi Patel, PhD | Open Curriculum — CC BY 4.0*

---

## Learning Objectives

By the end of this notebook, you will be able to:

- **Work with multiple LLM APIs** — OpenAI, Anthropic, and HuggingFace
- **Implement structured outputs** — JSON mode, function calling, and schema validation
- **Measure and compare** — cost, latency, and reliability across providers
- **Understand data privacy** — implications of API-based LLM usage and retention policies
- **Design production-ready integrations** — error handling, retries, rate limiting, and fallbacks

---
## ⚙️ Setup — Install & Import

In [ ]:
!pip install -q openai anthropic huggingface_hub requests tiktoken

In [ ]:
import os
import json
import time
import getpass
from typing import Optional, Tuple

import tiktoken
from openai import OpenAI
from anthropic import Anthropic
from huggingface_hub import InferenceClient

---
## 1. Working with LLM APIs

### 1.1 OpenAI Chat Completions API

OpenAI's API uses a chat-based interface. We use `getpass` to securely prompt for your API key — **never hardcode keys in notebooks or source code.**

In [ ]:
# Securely get API key — leave blank to use mock/fallback
openai_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (or press Enter to skip): ")

def call_openai(prompt: str, model: str = "gpt-4o-mini") -> str:
    """Call OpenAI Chat Completions API. Returns mock response if no key."""
    if not openai_key:
        return "[Mock] OpenAI response: This is a placeholder. Add your API key to get real outputs."
    client = OpenAI(api_key=openai_key)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [ ]:
prompt = "In one sentence, what is the capital of France?"
print("OpenAI:", call_openai(prompt))

### 1.2 Anthropic Claude API

Anthropic's Claude API uses a similar chat interface but with different parameter names and message structure.

In [ ]:
anthropic_key = os.environ.get("ANTHROPIC_API_KEY") or getpass.getpass("Anthropic API key (or press Enter to skip): ")

def call_anthropic(prompt: str, model: str = "claude-3-5-haiku-20241022") -> str:
    """Call Anthropic Claude API. Returns mock response if no key."""
    if not anthropic_key:
        return "[Mock] Anthropic response: This is a placeholder. Add your API key to get real outputs."
    client = Anthropic(api_key=anthropic_key)
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

print("Anthropic:", call_anthropic(prompt))

### 1.3 HuggingFace Inference API

HuggingFace offers free inference for many models. A token is required for some models; others work without authentication.

In [ ]:
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or getpass.getpass("HuggingFace token (or press Enter for public models): ")

def call_huggingface(prompt: str, model: str = "mistralai/Mistral-7B-Instruct-v0.2") -> str:
    """Call HuggingFace Inference API. Works with free tier; token improves rate limits."""
    try:
        client = InferenceClient(token=hf_token if hf_token else None)
        response = client.chat_completion(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"[HuggingFace Error] {str(e)} — Try a different model or add HF token."

print("HuggingFace:", call_huggingface(prompt))

### 1.4 Side-by-Side Comparison

Send the same prompt to all three providers and compare outputs.

In [ ]:
test_prompt = "List three benefits of renewable energy in bullet points."

results = {}
results["OpenAI"] = call_openai(test_prompt)
results["Anthropic"] = call_anthropic(test_prompt)
results["HuggingFace"] = call_huggingface(test_prompt)

for provider, output in results.items():
    print(f"\n--- {provider} ---\n{output}")

### 1.5 API Key Management Best Practices

| Practice | Why |
|----------|-----|
| Use `getpass` or env vars | Keys never appear in logs or screenshots |
| Never commit keys to git | Add `.env` to `.gitignore` |
| Rotate keys periodically | Limits blast radius if compromised |
| Use separate keys per environment | Dev/staging/prod isolation |
| Prefer env vars in production | `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. |

---
## 2. Structured Outputs

### 2.1 JSON Mode with OpenAI

Use `response_format={"type": "json_object"}` to force valid JSON output.

In [ ]:
def call_openai_json(prompt: str, model: str = "gpt-4o-mini") -> dict:
    """Call OpenAI with JSON mode. Returns parsed dict or empty dict on failure."""
    if not openai_key:
        return {"mock": True, "message": "Add API key for JSON mode"}
    client = OpenAI(api_key=openai_key)
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"},
    )
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"error": "Invalid JSON", "raw": response.choices[0].message.content}

json_prompt = "Return a JSON object with keys 'topic', 'summary', 'key_points' (array of 3 strings) for: 'Machine learning in healthcare'."
result = call_openai_json(json_prompt)
print(json.dumps(result, indent=2))

### 2.2 Function Calling / Tool Use Schema

Define tools the model can "call" — useful for structured extraction and agent workflows.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "extract_entities",
            "description": "Extract named entities from text",
            "parameters": {
                "type": "object",
                "properties": {
                    "persons": {"type": "array", "items": {"type": "string"}, "description": "Person names"},
                    "organizations": {"type": "array", "items": {"type": "string"}, "description": "Organization names"},
                    "locations": {"type": "array", "items": {"type": "string"}, "description": "Location names"},
                },
                "required": ["persons", "organizations", "locations"],
            },
        },
    }
]

def call_openai_with_tools(prompt: str) -> Optional[dict]:
    """Call OpenAI with function/tool schema. Returns parsed tool args if used."""
    if not openai_key:
        return None
    client = OpenAI(api_key=openai_key)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        tools=tools,
        tool_choice="auto",
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        args = json.loads(msg.tool_calls[0].function.arguments)
        return args
    return None

text = "Apple CEO Tim Cook announced a new facility in Austin, Texas. Microsoft and Google are also expanding."
extracted = call_openai_with_tools(f"Extract entities from: {text}")
print("Extracted:", json.dumps(extracted, indent=2) if extracted else "(No tool call — add API key)")

### 2.3 Parsing and Validating Structured Outputs

Always validate and handle malformed outputs gracefully.

In [ ]:
def parse_and_validate_json(raw: str, required_keys: list = None) -> Tuple[Optional[dict], Optional[str]]:
    """Parse JSON, validate required keys. Returns (data, error_msg)."""
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        return None, f"Invalid JSON: {e}"
    if required_keys:
        missing = [k for k in required_keys if k not in data]
        if missing:
            return data, f"Missing keys: {missing}"
    return data, None

# Example: handle malformed output
malformed = '{"topic": "AI", "summary": "Great"}'  # missing key_points
data, err = parse_and_validate_json(malformed, required_keys=["topic", "summary", "key_points"])
print("Data:", data)
print("Error:", err)

---
## 3. Cost, Latency, and Reliability

### 3.1 Token Counting with tiktoken

Accurate token counts are essential for cost estimation. tiktoken is OpenAI's tokenizer.

In [ ]:
def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Count tokens for a given model. Uses cl100k_base for GPT-4/3.5."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))

sample = "The quick brown fox jumps over the lazy dog. " * 10
print(f"Tokens: {count_tokens(sample)}")
print(f"Chars: {len(sample)}")

### 3.2 Cost Estimation

Prices vary by model. Example rates (as of 2024; check provider docs for current pricing).

In [ ]:
# Example prices per 1M tokens (input / output) — update from provider docs
PRICES = {
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4o": (2.50, 10.00),
    "claude-3-5-haiku": (0.80, 4.00),
}

def estimate_cost(input_tokens: int, output_tokens: int, model: str) -> float:
    """Estimate cost in USD."""
    if model not in PRICES:
        return 0.0
    in_price, out_price = PRICES[model]
    return (input_tokens * in_price + output_tokens * out_price) / 1_000_000

prompt_tokens = count_tokens("Summarize this article.")
output_tokens = 150  # typical summary length
cost = estimate_cost(prompt_tokens, output_tokens, "gpt-4o-mini")
print(f"Estimated cost: ${cost:.6f}")

### 3.3 Latency Measurement

Time API calls to compare provider responsiveness.

In [ ]:
def measure_latency(provider: str, prompt: str) -> Tuple[str, float]:
    """Return (response, latency_seconds)."""
    start = time.perf_counter()
    if provider == "OpenAI":
        resp = call_openai(prompt)
    elif provider == "Anthropic":
        resp = call_anthropic(prompt)
    elif provider == "HuggingFace":
        resp = call_huggingface(prompt)
    else:
        return "", 0.0
    elapsed = time.perf_counter() - start
    return resp, elapsed

p = "What is 2+2? Reply in one word."
for prov in ["OpenAI", "Anthropic", "HuggingFace"]:
    resp, lat = measure_latency(prov, p)
    print(f"{prov}: {lat:.2f}s — {resp[:60]}..." if len(resp) > 60 else f"{prov}: {lat:.2f}s — {resp}")

### 3.4 Reliability Benchmarking

Simple benchmark with retry logic and error handling.

In [ ]:
def benchmark_provider(provider: str, prompt: str, n_trials: int = 3) -> dict:
    """Run n_trials, track success rate, avg latency, errors."""
    latencies = []
    errors = []
    for _ in range(n_trials):
        try:
            resp, lat = measure_latency(provider, prompt)
            if resp and "[Mock]" not in resp and "Error" not in resp:
                latencies.append(lat)
            else:
                errors.append(resp[:80])
        except Exception as e:
            errors.append(str(e)[:80])
    return {
        "provider": provider,
        "success_rate": len(latencies) / n_trials if n_trials else 0,
        "avg_latency_s": sum(latencies) / len(latencies) if latencies else None,
        "errors": errors,
    }

for prov in ["OpenAI", "Anthropic", "HuggingFace"]:
    r = benchmark_provider(prov, "Say hello.", n_trials=2)
    print(r)

---
## 4. Data Privacy and Governance

### What Data Gets Sent to the API Provider?

When you call a cloud LLM API, you send:
- **Your prompts** (user input, system instructions)
- **Model responses** (returned to you)
- **Metadata** (model name, tokens, timestamps)

This data may be:
- **Logged** for debugging and abuse prevention
- **Retained** for varying periods (see provider policies)
- **Used for training** unless you opt out (OpenAI, Anthropic offer enterprise opt-outs)

### Data Retention Policies (as of 2024)

| Provider | Default Retention | Training Use |
|----------|-------------------|-------------|
| OpenAI | 30 days (API) | Opt-out available |
| Anthropic | 90 days | Opt-out available |
| HuggingFace | Varies by endpoint | Check terms |

### When to Use API vs. Self-Hosted

- **API**: Fast iteration, managed infra, latest models. Use when data sensitivity allows.
- **Self-hosted**: Full control, no data leaves your infra. Use for HIPAA, legal, financial, or high-sensitivity data.

### Logging and Auditing

Always log API usage for cost tracking, debugging, and compliance. **Redact sensitive data** before logging.

In [ ]:
import re

SENSITIVE_PATTERNS = [
    (re.compile(r"\b\d{3}-\d{2}-\d{4}\b"), "[SSN_REDACTED]"),
    (re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"), "[EMAIL_REDACTED]"),
    (re.compile(r"\b\d{16}\b"), "[CARD_REDACTED]"),
]

def redact(text: str) -> str:
    """Redact common PII before logging."""
    for pat, repl in SENSITIVE_PATTERNS:
        text = pat.sub(repl, text)
    return text

def log_api_call(provider: str, prompt: str, response: str, tokens_in: int, tokens_out: int, cost: float):
    """Log API call with redacted prompts/responses."""
    entry = {
        "provider": provider,
        "prompt_redacted": redact(prompt)[:200] + "..." if len(prompt) > 200 else redact(prompt),
        "response_redacted": redact(response)[:200] + "..." if len(response) > 200 else redact(response),
        "tokens_in": tokens_in,
        "tokens_out": tokens_out,
        "cost_usd": cost,
    }
    print(json.dumps(entry, indent=2))
    # In production: write to file, DB, or logging service

log_api_call("OpenAI", "User email: john@example.com asked about SSN 123-45-6789", "Response here", 20, 50, 0.0001)

---
## 5. Production Patterns

### 5.1 Retry with Exponential Backoff

In [ ]:
def call_with_retry(call_fn, *args, max_retries: int = 3, **kwargs):
    """Retry with exponential backoff on transient errors."""
    last_err = None
    for attempt in range(max_retries):
        try:
            return call_fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            if "rate" in str(e).lower() or "429" in str(e) or "timeout" in str(e).lower():
                wait = 2 ** attempt
                print(f"Retry {attempt+1}/{max_retries} after {wait}s: {e}")
                time.sleep(wait)
            else:
                raise
    raise last_err

### 5.2 Rate Limiting, Caching, and Fallback

| Pattern | Purpose |
|---------|--------|
| **Rate limiting** | Respect provider limits; use queues or token buckets |
| **Caching** | Cache identical prompts to reduce cost and latency |
| **Fallback** | If primary provider fails, try secondary (e.g., OpenAI → Anthropic) |

In [ ]:
CACHE = {}

def call_with_cache(prompt: str, provider: str = "OpenAI") -> str:
    """Simple in-memory cache. Key = hash(prompt + provider)."""
    key = hash((prompt, provider))
    if key in CACHE:
        return CACHE[key]
    resp = call_openai(prompt) if provider == "OpenAI" else call_anthropic(prompt) if provider == "Anthropic" else call_huggingface(prompt)
    CACHE[key] = resp
    return resp

def call_with_fallback(prompt: str) -> str:
    """Try OpenAI, then Anthropic, then HuggingFace."""
    for prov, fn in [("OpenAI", call_openai), ("Anthropic", call_anthropic), ("HuggingFace", call_huggingface)]:
        try:
            r = fn(prompt)
            if r and "[Mock]" not in r and "Error" not in r:
                return r
        except Exception:
            continue
    return "All providers failed or unavailable."

print("Fallback demo:", call_with_fallback("What is the capital of Japan?")[:80])

---
## 6. Exercises

### Exercise 1: Multi-Provider Comparison

Send the same **5 prompts** to OpenAI and HuggingFace. Compare outputs, latency, and estimated cost. Create a small table or summary.

In [ ]:
# Your code here
PROMPTS = [
    "What is machine learning?",
    "Name three programming languages.",
    "Explain recursion in one sentence.",
    "What is the speed of light?",
    "Summarize the benefits of exercise.",
]
# For each prompt: call OpenAI and HuggingFace, measure latency, estimate cost, compare outputs

### Exercise 2: Structured Entity Extraction

Build a pipeline that extracts **named entities** (persons, organizations, locations) from text into a JSON schema. Use JSON mode or function calling. Handle malformed outputs.

In [ ]:
# Your code here
# Input: raw text
# Output: {"persons": [...], "organizations": [...], "locations": [...]}
# Include validation and fallback for malformed JSON

### Exercise 3: Production-Ready API Wrapper

Implement a wrapper that includes:
- Retry with exponential backoff
- Logging (with redaction)
- Cost tracking
- Optional fallback to a secondary provider

In [ ]:
# Your code here
# class ProductionLLMClient:
#     def __init__(self, primary="OpenAI", fallback="Anthropic"): ...
#     def complete(self, prompt): ...  # retry, log, track cost, fallback

---
## 7. Responsible AI Reflection

When you use a cloud LLM API, your users' data travels to a third-party server. The provider may log it, use it for training, or store it indefinitely.

**Consider:**
- If you were building an AI system for a **hospital**, a **law firm**, or a **school**, which providers would you trust with that data?
- What would you need to **verify** before making that decision?
- Is "read the terms of service" a sufficient answer?

*Write a short reflection (2–3 paragraphs) in the cell below.*

In [ ]:
# Your reflection here
reflection = """

"""
print(reflection)

---
## 8. Summary & Next Steps

### Summary

- **Multiple APIs**: OpenAI, Anthropic, HuggingFace — each with different strengths and pricing
- **Structured outputs**: JSON mode and function calling for reliable parsing
- **Cost & latency**: Token counting, cost estimation, benchmarking
- **Privacy**: Understand retention, opt-outs, and when to self-host
- **Production**: Retries, caching, fallbacks, logging with redaction

### Next Steps

- **Week 6**: RAG (Retrieval-Augmented Generation) — grounding LLMs with your own data
- Explore provider-specific features (e.g., OpenAI Assistants, Anthropic tool use)
- Set up a local LLM (Ollama, vLLM) for development without API costs